# CBMS — Active Learning Pipeline

This notebook implements **two completely separate workflows**. Do not conflate them.

---

## Workflow A — Confirmed directory retraining (no human review needed)

Your hand-sorted ground-truth clips live in the original dataset under `spitting/`, `littering/`, `normal/`. These clips are **already labeled**. The model does not re-examine them. They are copied directly into the `confirmed/` fine-tuning pool and immediately used for training.

```
Workflow A:
  SOURCE_DATA_ROOT/
    spitting/   ← ground truth (label = spitting)
    littering/  ← ground truth (label = littering)
    normal/     ← ground truth (label = normal)
          │
          └─► copied into confirmed/<label>/  →  fine-tune
```

Run order for Workflow A: **Cell 1 → restart → Cells 2–5 → Cell 5b → Cells 7–9**

---

## Workflow B — Raw footage review (human-in-the-loop)

Feed in raw CCTV or any footage. The model detects behaviours, saves short evidence clips to `review/`. You watch each clip and give feedback via **directory drag-and-drop** — no coding required.

```
Workflow B:
  RAW_FOOTAGE_DIR/*.mp4
          │
          ▼  (YOLO + GRU detection)
  review/<predicted_label>/clip_NNNN.mp4
          │
          ├─ Correct detection  →  drag to  confirmed/<label>/
          │                         label = same as predicted
          │                         → fine-tune as TP
          │
          └─ Wrong detection    →  drag to  wrong_calls/
                                   → automatically ingested as FP
                                     (relabelled normal, weighted x2)
                                   → fine-tune runs immediately
```

Run order for Workflow B: Cells 2–5 → **Cell 6** (detect on raw footage) → label clips in file explorer → **Cell 6b** (ingest wrong_calls) → **Cells 7–9** (fine-tune) → **Cell 9b** (TP regression test)

---

> **Key invariant:** `confirmed/` is the single ground-truth fine-tuning pool, fed from both workflows. The model never re-runs on confirmed clips — it only trains on them.

## Cell 1 — Install
Run once, then restart the session.

In [ ]:
import subprocess, sys
pkgs = [
    "mediapipe==0.10.14",
    "ultralytics>=8.3.100",
    "opencv-python-headless",
    "lapx>=0.5.2",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Done. Restart the session now, then run from Cell 2.")

## Cell 2 — Imports

In [ ]:
import os
import cv2
import json
import random
import shutil
import numpy as np
from pathlib import Path
from collections import defaultdict, deque, Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from ultralytics import YOLO
import mediapipe as mp

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Cell 3 — Configuration

> **Thresholds must stay in sync with the production pipeline notebook.**  
> `WATCH_WINDOW`, `YOLO_CONF`, `ACTIVITY_CONFIDENCE`, `ACTIVITY_SAMPLE_EVERY`, and `CONFIRMATION_VOTES` should be identical in both notebooks so the active learning data reflects real deployment conditions.

### Path layout

```
labeled/
├── review/
│   ├── spitting/      ← Workflow B: model fired on raw footage, awaiting your decision
│   └── littering/     ← Workflow B: model fired on raw footage, awaiting your decision
├── wrong_calls/       ← Workflow B: drag FP clips here — these become normal corrections
└── confirmed/
    ├── spitting/      ← ground truth TPs (from both workflows)
    ├── littering/     ← ground truth TPs (from both workflows)
    └── normal/        ← ground truth normals + FP corrections from wrong_calls/
```

In [ ]:
# ── Workflow A: ground-truth source (already labeled, never re-detected) ──────
# Root of uploaded dataset containing spitting/, littering/, normal/
SOURCE_DATA_ROOT = "/kaggle/input/datasets/gallimus/cbms-training-data"

# ── Workflow B: raw footage to run detection on ───────────────────────────────
# Point this at any folder of raw CCTV/phone videos you want the model to scan.
# These must NOT be the already-labeled clips from SOURCE_DATA_ROOT.
RAW_FOOTAGE_DIR = "/kaggle/input/datasets/gallimus/raw-cctv-footage"

# ── Model paths ───────────────────────────────────────────────────────────────
CLASSIFIER_PATH  = "/kaggle/input/datasets/gallimus/pretrained-cbms-classifier-model/activity_classifier.pt"

# ── Working directory layout ──────────────────────────────────────────────────
LABEL_ROOT     = "/kaggle/working/labeled"
REVIEW_ROOT    = os.path.join(LABEL_ROOT, "review")       # Workflow B: pending human review
WRONG_CALLS    = os.path.join(LABEL_ROOT, "wrong_calls")  # Workflow B: drag FPs here
CONFIRMED_ROOT = os.path.join(LABEL_ROOT, "confirmed")   # fine-tuning pool (both workflows)

# ── Pipeline thresholds — keep identical to production pipeline ───────────────
YOLO_CONF             = 0.45
ACTIVITY_CONFIDENCE   = 0.90
ACTIVITY_SAMPLE_EVERY = 8
WATCH_WINDOW          = 32    # must equal N_FRAMES from training
CONFIRMATION_VOTES    = 3

# ── Clip extraction ───────────────────────────────────────────────────────────
CLIP_PRE_SEC  = 2.0
CLIP_POST_SEC = 3.0

# ── Fine-tuning ───────────────────────────────────────────────────────────────
FT_EPOCHS        = 50
FT_BATCH_SIZE    = 16
FT_LR            = 5e-4
FT_VAL_SPLIT     = 0.20
FT_EARLY_STOP    = 12
FT_LR_PATIENCE   = 6
FP_WEIGHT_FACTOR = 2     # repeat wrong_calls clips N times as hard negatives
MINORITY_COPIES  = 2     # extra augmented copies for minority classes

# ── Workflow B post-retrain TP test ──────────────────────────────────────────
# After fine-tuning, the model is run on TP clips (those it got right) to check
# if it still recognises them after the update. If it fails on them, they are
# flagged for another fine-tune round.
TP_REGRESSION_CONFIDENCE = 0.70   # below this → regression flagged

# ── Model architecture (must match training notebook) ─────────────────────────
N_KEYPOINTS = 33
FEATURES    = N_KEYPOINTS * 3   # 99

print("Config loaded.")
print(f"  SOURCE_DATA_ROOT  → {SOURCE_DATA_ROOT}")
print(f"  RAW_FOOTAGE_DIR   → {RAW_FOOTAGE_DIR}")
print(f"  REVIEW_ROOT       → {REVIEW_ROOT}")
print(f"  WRONG_CALLS       → {WRONG_CALLS}")
print(f"  CONFIRMED_ROOT    → {CONFIRMED_ROOT}")

## Cell 4 — Model definition & load

Bidirectional GRU — matches the training notebook exactly.

In [ ]:
class PoseActivityClassifier(nn.Module):
    """
    Bidirectional 2-layer GRU with mean pooling.
    Architecture must be identical to the training notebook.
    """
    def __init__(self, input_size, hidden1, hidden2, num_classes, dropout=0.0):
        super().__init__()
        self.gru1       = nn.GRU(input_size,  hidden1, batch_first=True, bidirectional=True)
        self.gru2       = nn.GRU(hidden1 * 2, hidden2, batch_first=True, bidirectional=True)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden2 * 2, num_classes)

    def forward(self, x):
        out, _ = self.gru1(x)
        out, _ = self.gru2(out)
        out    = torch.mean(out, dim=1)
        return self.classifier(self.dropout(out))


print("Loading classifier...")
ckpt = torch.load(CLASSIFIER_PATH, map_location=DEVICE, weights_only=False)

# Verify the checkpoint is the GRU version
arch = ckpt.get("architecture", "unknown")
if arch != "bidirectional_gru":
    raise RuntimeError(
        f"Checkpoint architecture is '{arch}', expected 'bidirectional_gru'. "
        "Upload the model produced by the updated training notebook."
    )

CLASS_NAMES = ckpt["class_names"]
N_FRAMES    = ckpt["n_frames"]    # should be 32

if N_FRAMES != WATCH_WINDOW:
    print(f"[WARN] Checkpoint n_frames={N_FRAMES} but WATCH_WINDOW={WATCH_WINDOW}. "
          f"Setting WATCH_WINDOW={N_FRAMES}.")
    WATCH_WINDOW = N_FRAMES

CLASSIFIER = PoseActivityClassifier(
    ckpt["features"], ckpt["hidden1"], ckpt["hidden2"],
    len(CLASS_NAMES), dropout=0.0,
).to(DEVICE)
CLASSIFIER.load_state_dict(ckpt["model_state"])
CLASSIFIER.eval()

print(f"  Architecture : {arch}")
print(f"  Classes      : {CLASS_NAMES}")
print(f"  n_frames     : {N_FRAMES}")
print(f"  Val accuracy : {ckpt.get('val_accuracy', 0):.1%}")

print("\nLoading YOLO...")
YOLO_MODEL = YOLO("yolov8n.pt")
if torch.cuda.is_available():
    YOLO_MODEL.to("cuda")
    print("  YOLO -> GPU")

print("\nLoading MediaPipe Pose...")
MP_POSE = mp.solutions.pose.Pose(
    static_image_mode=False,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.3,
)
print("All models loaded.")

## Cell 5 — Shared helpers

### Key fix: nose-relative keypoints
Keypoints are zero-centred relative to the nose landmark before storage. This **must** match the training notebook's preprocessing — omitting it would cause the model to see features outside the distribution it was trained on.

In [ ]:
VID_EXTS = {".mp4", ".avi", ".mov", ".mkv"}


def extract_keypoints(crop: np.ndarray) -> np.ndarray:
    """
    Extract a 99-d nose-relative pose vector from a person crop.
    Returns zeros if MediaPipe finds no landmarks.
    """
    if crop is None or crop.size == 0:
        return np.zeros(FEATURES, dtype=np.float32)
    try:
        small  = cv2.resize(crop, (128, 128))
        rgb    = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
        result = MP_POSE.process(rgb)
        if result.pose_landmarks:
            kp   = result.pose_landmarks.landmark
            nose = kp[0]
            return np.array(
                [[lm.x - nose.x, lm.y - nose.y, lm.z] for lm in kp],
                dtype=np.float32,
            ).flatten()
    except Exception:
        pass
    return np.zeros(FEATURES, dtype=np.float32)


def classify_keypoints(kp_buffer: deque) -> tuple:
    """Run the GRU on the current keypoint buffer. Returns (activity, confidence)."""
    if len(kp_buffer) < WATCH_WINDOW:
        return "normal", 0.0
    seq  = np.stack(list(kp_buffer)[-WATCH_WINDOW:])
    x    = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(CLASSIFIER(x), dim=1).cpu().numpy()[0]
    best_idx = int(probs.argmax())
    return CLASS_NAMES[best_idx], round(float(probs[best_idx]), 4)


def classify_clip_file(clip_path: str) -> tuple:
    """
    Run the current CLASSIFIER on a saved clip file.
    Returns (predicted_activity, confidence, all_probs_dict).
    Used in post-retrain TP regression testing.
    """
    cap = cv2.VideoCapture(clip_path)
    buffer = deque(maxlen=WATCH_WINDOW)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Use full frame as crop — these clips are already tight
        buffer.append(extract_keypoints(frame))
    cap.release()

    if len(buffer) < WATCH_WINDOW:
        return "normal", 0.0, {}

    seq = np.stack(list(buffer)[-WATCH_WINDOW:])
    x   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(CLASSIFIER(x), dim=1).cpu().numpy()[0]
    best_idx = int(probs.argmax())
    probs_dict = {CLASS_NAMES[i]: round(float(probs[i]), 4) for i in range(len(CLASS_NAMES))}
    return CLASS_NAMES[best_idx], round(float(probs[best_idx]), 4), probs_dict


def extract_clip_around(video_path: str, event_frame: int, src_fps: float,
                        pre_sec: float, post_sec: float, output_path: str):
    """Save a short clip centred on event_frame to output_path."""
    cap   = cv2.VideoCapture(video_path)
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start = max(0,     int(event_frame - pre_sec  * src_fps))
    end   = min(total, int(event_frame + post_sec * src_fps))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        src_fps, (w, h),
    )
    for _ in range(end - start):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
    cap.release()
    out.release()


def count_clips_in(directory: str) -> int:
    if not os.path.exists(directory):
        return 0
    return sum(1 for f in Path(directory).rglob("*")
               if f.suffix.lower() in VID_EXTS)


def ensure_directory_layout():
    """Create the full working directory tree."""
    for subdir in CLASS_NAMES + ["normal"]:
        Path(os.path.join(REVIEW_ROOT,    subdir)).mkdir(parents=True, exist_ok=True)
        Path(os.path.join(CONFIRMED_ROOT, subdir)).mkdir(parents=True, exist_ok=True)
    Path(WRONG_CALLS).mkdir(parents=True, exist_ok=True)
    print(f"Directory layout ready under {LABEL_ROOT}/")
    print(f"  review/         ← Workflow B clips awaiting your decision")
    print(f"  wrong_calls/    ← drag FP clips here (they become normal corrections)")
    print(f"  confirmed/      ← fine-tuning pool (both workflows feed here)")
    for subdir in CLASS_NAMES:
        print(f"    confirmed/{subdir}/")
    print(f"    confirmed/normal/   ← FP corrections + ground-truth normals")


ensure_directory_layout()
print("\nHelpers defined.")

## Cell 5b — Workflow A: bootstrap confirmed/ from ground-truth source

Copies your already-labeled `SOURCE_DATA_ROOT/spitting/`, `/littering/`, `/normal/` clips directly into `confirmed/` with a `src_` prefix so they can never be confused with Workflow B clips.

**This cell is idempotent** — re-running it is safe.

> The model is never run on these clips. They go straight into fine-tuning as-is.

In [ ]:
def bootstrap_confirmed_from_source(source_root, confirmed_root):
    """
    Workflow A: copy ground-truth clips from the original labeled dataset
    directly into confirmed/<label>/ without running any detection on them.

    Files are prefixed src_ to distinguish them from Workflow B clips.
    Already-copied files are skipped (idempotent).
    """
    source_path    = Path(source_root)
    confirmed_path = Path(confirmed_root)
    total_copied   = 0
    total_skipped  = 0

    print(f"[Workflow A] Anchoring ground-truth data from {source_root} ...")
    for class_dir in sorted(source_path.iterdir()):
        if not class_dir.is_dir():
            continue
        label    = class_dir.name
        dest_dir = confirmed_path / label
        dest_dir.mkdir(parents=True, exist_ok=True)

        videos = sorted([f for f in class_dir.rglob("*")
                         if f.suffix.lower() in VID_EXTS])
        copied  = 0
        skipped = 0
        for vid in videos:
            dest_file = dest_dir / f"src_{vid.name}"
            if dest_file.exists():
                skipped += 1
            else:
                shutil.copy2(vid, dest_file)
                copied += 1
        print(f"  [{label:<12}]  {copied} copied  {skipped} already present  "
              f"({len(videos)} total)")
        total_copied  += copied
        total_skipped += skipped

    print(f"\nDone. {total_copied} new files copied, {total_skipped} skipped.")
    print("confirmed/ is now seeded with Workflow A ground-truth data.")


bootstrap_confirmed_from_source(SOURCE_DATA_ROOT, CONFIRMED_ROOT)

## Cell 6 — Workflow B: detect behaviours in raw footage

Runs YOLO + ByteTrack + GRU on every video in `RAW_FOOTAGE_DIR`.

**This cell does NOT touch `confirmed/`.**  
All output goes to `review/<predicted_activity>/`. After this cell finishes:

1. Open your file explorer and browse `review/`.
2. Watch each clip.
3. **Correct detection** → drag clip to `confirmed/<true_label>/` (use the predicted label folder if it really is that behaviour).
4. **Wrong detection** → drag clip to `wrong_calls/`.
5. Run Cell 6b to ingest the wrong_calls, then Cells 7–9 to fine-tune.

In [ ]:
# ── Check that RAW_FOOTAGE_DIR is not the same as SOURCE_DATA_ROOT ────────────
if os.path.realpath(RAW_FOOTAGE_DIR) == os.path.realpath(SOURCE_DATA_ROOT):
    raise RuntimeError(
        "RAW_FOOTAGE_DIR points to the same location as SOURCE_DATA_ROOT.\n"
        "Workflow B must operate on raw, unlabeled footage — not your ground-truth dataset.\n"
        "Update RAW_FOOTAGE_DIR in Cell 3."
    )

raw_video_paths = [
    str(p) for p in Path(RAW_FOOTAGE_DIR).rglob("*")
    if p.suffix.lower() in VID_EXTS
] if os.path.exists(RAW_FOOTAGE_DIR) else []

if not raw_video_paths:
    print(f"[Workflow B] No videos found in RAW_FOOTAGE_DIR: {RAW_FOOTAGE_DIR}")
    print("Update RAW_FOOTAGE_DIR in Cell 3 and re-run.")
else:
    print(f"[Workflow B] Found {len(raw_video_paths)} videos to scan.")
    print(f"  Output clips → {REVIEW_ROOT}")
    print()

    bytetrack_cfg = """
tracker_type: bytetrack
track_high_thresh: 0.35
track_low_thresh: 0.05
new_track_thresh: 0.35
track_buffer: 60
match_thresh: 0.70
fuse_score: True
"""
    BT_CFG_PATH = "/kaggle/working/bytetrack_custom.yaml"
    with open(BT_CFG_PATH, "w") as f:
        f.write(bytetrack_cfg)

    clip_index   = 0
    total_events = 0

    # Registry of TP clips: maps clip_path -> predicted_label
    # Used later by Cell 9b (post-retrain regression test).
    TP_REVIEW_REGISTRY_PATH = "/kaggle/working/tp_review_registry.json"
    tp_registry = {}  # will be populated as user confirms TPs into confirmed/

    for video_path in raw_video_paths:
        if not os.path.exists(video_path):
            print(f"SKIP — not found: {video_path}")
            continue

        cap      = cv2.VideoCapture(video_path)
        src_fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        print(f"\nProcessing: {os.path.basename(video_path)}  "
              f"({n_frames} frames @ {src_fps:.1f} fps)")

        cap           = cv2.VideoCapture(video_path)
        kp_buffers    = defaultdict(lambda: deque(maxlen=WATCH_WINDOW))
        vote_counters = defaultdict(lambda: {"activity": "", "count": 0})
        fired_events  = set()   # (tid, bucket) — suppress duplicate clips
        frame_idx     = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            h_f, w_f = frame.shape[:2]

            try:
                results = YOLO_MODEL.track(
                    frame, conf=YOLO_CONF, classes=[0], persist=True,
                    tracker=BT_CFG_PATH, verbose=False,
                )[0]
            except Exception as e:
                print(f"  [WARN] YOLO error at frame {frame_idx}: {e}")
                frame_idx += 1
                continue

            if results.boxes is not None and results.boxes.id is not None:
                for box in results.boxes:
                    try:
                        tid          = int(box.id[0])
                        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
                        x1, y1       = max(0, x1), max(0, y1)
                        x2, y2       = min(w_f, x2), min(h_f, y2)
                        if x2 <= x1 or y2 <= y1:
                            continue
                        crop = frame[y1:y2, x1:x2].copy()
                    except Exception:
                        continue

                    kp_buffers[tid].append(extract_keypoints(crop))

                    if (frame_idx % ACTIVITY_SAMPLE_EVERY == 0
                            and len(kp_buffers[tid]) == WATCH_WINDOW):

                        activity, conf = classify_keypoints(kp_buffers[tid])

                        if activity != "normal" and conf >= ACTIVITY_CONFIDENCE:
                            vc = vote_counters[tid]
                            if vc["activity"] == activity:
                                vc["count"] += 1
                            else:
                                vote_counters[tid] = {"activity": activity, "count": 1}

                            if vote_counters[tid]["count"] >= CONFIRMATION_VOTES:
                                vote_counters[tid] = {"activity": "", "count": 0}

                                # One clip per track per 2-second window
                                event_key = (tid, frame_idx // int(src_fps * 2))
                                if event_key in fired_events:
                                    continue
                                fired_events.add(event_key)

                                # Save clip into review/<activity>/
                                clip_name = (
                                    f"clip_{clip_index:04d}_{activity}"
                                    f"_t{tid}_f{frame_idx}.mp4"
                                )
                                clip_out = os.path.join(REVIEW_ROOT, activity, clip_name)
                                extract_clip_around(
                                    video_path, frame_idx, src_fps,
                                    CLIP_PRE_SEC, CLIP_POST_SEC, clip_out,
                                )

                                # Metadata JSON alongside the clip
                                meta = {
                                    "clip_file":       clip_name,
                                    "predicted_label": activity,
                                    "model_conf":      conf,
                                    "source_video":    video_path,
                                    "frame_index":     frame_idx,
                                    "timestamp_s":     round(frame_idx / src_fps, 2),
                                    "track_id":        tid,
                                    "workflow":        "B",
                                }
                                meta_out = clip_out.replace(".mp4", ".json")
                                with open(meta_out, "w") as mf:
                                    json.dump(meta, mf, indent=2)

                                clip_index   += 1
                                total_events += 1
                                print(f"  [{clip_index:04d}] {activity} conf={conf:.0%} "
                                      f"track={tid} t={frame_idx/src_fps:.1f}s")
                        else:
                            vote_counters[tid] = {"activity": "", "count": 0}

            if frame_idx % 300 == 0 and frame_idx > 0:
                print(f"  ... {frame_idx}/{n_frames} frames "
                      f"({frame_idx/n_frames*100:.0f}%)")

            frame_idx += 1

        cap.release()

    print(f"\n{'='*55}")
    print(f"[Workflow B] Detection complete. {total_events} clips saved.")
    print(f"\nNext steps:")
    print(f"  1. Open file explorer → browse {REVIEW_ROOT}")
    print(f"  2. Watch each clip")
    print(f"  3. Correct detection  → drag to  confirmed/<true_label>/")
    print(f"  4. Wrong detection    → drag to  wrong_calls/")
    print(f"  5. Run Cell 6b to ingest wrong_calls")
    print(f"  6. Run Cells 7–9 to fine-tune")

## Cell 6b — Workflow B: ingest wrong_calls/ as FP corrections

After you have dragged false-positive clips from `review/` into `wrong_calls/`, run this cell.

What it does:
- Reads every video file in `wrong_calls/`.
- **Relabels them as `normal`** (because the model fired incorrectly — these are not civic offences).
- Copies them into `confirmed/normal/` with a `fp_` prefix.
- Applies `FP_WEIGHT_FACTOR` repeat weight during training (set in Cell 3) so the model treats them as hard negatives.
- Logs each ingested clip to `fp_ingest_log.json`.
- **Does not delete** the original files in `wrong_calls/` — you can keep them as an audit trail.

This cell is **idempotent**: re-running it is safe (already-ingested clips are detected via the `fp_` prefix and skipped).

After running, proceed to **Cells 7–9** (fine-tune immediately in this session).

In [ ]:
FP_INGEST_LOG    = "/kaggle/working/fp_ingest_log.json"
normal_dest_dir  = Path(CONFIRMED_ROOT) / "normal"
normal_dest_dir.mkdir(parents=True, exist_ok=True)

wrong_clips = sorted([
    f for f in Path(WRONG_CALLS).rglob("*")
    if f.suffix.lower() in VID_EXTS
])

if not wrong_clips:
    print(f"[Workflow B] wrong_calls/ is empty.")
    print(f"  Drag false-positive clips from review/ into:")
    print(f"  {WRONG_CALLS}")
    print(f"  Then re-run this cell.")
else:
    # Load existing ingest log
    ingest_log = []
    if os.path.exists(FP_INGEST_LOG):
        with open(FP_INGEST_LOG) as f:
            ingest_log = json.load(f)
    already_ingested = {entry["source"] for entry in ingest_log}

    new_count = 0
    skip_count = 0

    print(f"[Workflow B] Ingesting {len(wrong_clips)} clips from wrong_calls/ as FP corrections...")
    print(f"  Destination: {normal_dest_dir}")
    print(f"  Weight factor during training: x{FP_WEIGHT_FACTOR}")
    print()

    for clip in wrong_clips:
        clip_key = str(clip)
        if clip_key in already_ingested:
            skip_count += 1
            continue

        dest_name = f"fp_{clip.name}"
        dest_path = normal_dest_dir / dest_name

        # Handle name collision (shouldn't happen with fp_ prefix but be safe)
        if dest_path.exists():
            stem   = clip.stem
            suffix = clip.suffix
            dest_path = normal_dest_dir / f"fp_{stem}_{random.randint(1000,9999)}{suffix}"

        shutil.copy2(clip, dest_path)

        # Try to read original metadata json if it exists alongside the clip
        meta_src = clip.with_suffix(".json")
        original_meta = {}
        if meta_src.exists():
            try:
                with open(meta_src) as mf:
                    original_meta = json.load(mf)
            except Exception:
                pass

        log_entry = {
            "source":         clip_key,
            "dest":           str(dest_path),
            "ingested_label": "normal",
            "fp_weight":      FP_WEIGHT_FACTOR,
            "original_meta":  original_meta,
        }
        ingest_log.append(log_entry)
        already_ingested.add(clip_key)
        new_count += 1
        print(f"  [FP] {clip.name}  →  confirmed/normal/{dest_path.name}")

    # Save updated log
    with open(FP_INGEST_LOG, "w") as f:
        json.dump(ingest_log, f, indent=2)

    print(f"\n[Workflow B] Ingest complete.")
    print(f"  {new_count} new FP clips ingested into confirmed/normal/")
    print(f"  {skip_count} already ingested (skipped)")
    print(f"  Log saved → {FP_INGEST_LOG}")
    print(f"\nconfirmed/normal/ now contains:")
    total_normal = count_clips_in(str(normal_dest_dir))
    print(f"  {total_normal} total clips (ground-truth + FP corrections)")
    print(f"\nRun Cells 7–9 to fine-tune now.")

## Cell 7 — Labeling status check

Run this **after** you've moved clips into `confirmed/` (and run Cell 6b for wrong_calls) to see a summary of what's ready for retraining. Safe to run multiple times.

In [ ]:
# Count what's still in review (unlabeled)
print("Unlabeled — still in review/ (awaiting your decision):")
total_review = 0
if os.path.exists(REVIEW_ROOT):
    for activity in sorted(os.listdir(REVIEW_ROOT)):
        n = count_clips_in(os.path.join(REVIEW_ROOT, activity))
        if n:
            print(f"  review/{activity:<15}  {n} clips")
            total_review += n
if total_review == 0:
    print("  (none — all Workflow B clips have been reviewed)")

print()
print(f"In wrong_calls/ (FP, awaiting Cell 6b ingest):")
wc_count = count_clips_in(WRONG_CALLS)
print(f"  {wc_count} clips" if wc_count else "  (empty)")

print()
print("Fine-tuning pool — confirmed/:")
total_confirmed = 0
if os.path.exists(CONFIRMED_ROOT):
    for true_label in sorted(os.listdir(CONFIRMED_ROOT)):
        n = count_clips_in(os.path.join(CONFIRMED_ROOT, true_label))
        if n:
            src_n = sum(
                1 for f in Path(os.path.join(CONFIRMED_ROOT, true_label)).rglob("*")
                if f.suffix.lower() in VID_EXTS and f.name.startswith("src_")
            )
            fp_n = sum(
                1 for f in Path(os.path.join(CONFIRMED_ROOT, true_label)).rglob("*")
                if f.suffix.lower() in VID_EXTS and f.name.startswith("fp_")
            )
            other_n = n - src_n - fp_n
            print(f"  confirmed/{true_label:<15}  {n} clips  "
                  f"(A:{src_n} ground-truth  B-TP:{other_n} confirmed  B-FP:{fp_n} corrections)")
            total_confirmed += n
if total_confirmed == 0:
    print("  (empty — run Cell 5b for Workflow A, or label Workflow B clips)")

print()
print(f"Total awaiting review : {total_review}")
print(f"Total ready to train  : {total_confirmed}")

if wc_count > 0:
    print(f"\n[!] {wc_count} clips in wrong_calls/ not yet ingested.")
    print("    Run Cell 6b first, then Cells 7–9.")
elif total_confirmed > 0:
    print("\nReady to retrain. Run Cell 8.")
else:
    print("\nNothing to train on yet.")

## Cell 8 — Build training dataset from confirmed/

### How confirmed clips map to training labels

| Clip prefix | Source | Training label | Weight |
|---|---|---|---|
| `src_*` | Workflow A ground-truth | label = folder name | ×1 |
| (no prefix) | Workflow B confirmed TP | label = folder name | ×1 |
| `fp_*` | Workflow B FP correction | always `normal` | ×`FP_WEIGHT_FACTOR` |

### Extraction strategy

| Video length | Strategy |
|---|---|
| ≤ `SHORT_CLIP_THRESHOLD_S` sec | Evenly-sampled (1 window — whole clip is one action) |
| > `SHORT_CLIP_THRESHOLD_S` sec | Sliding window (action could be anywhere) |

In [ ]:
SHORT_CLIP_THRESHOLD_S = 15.0
SLIDING_OVERLAP        = N_FRAMES // 2   # 50% overlap for long videos


# ---- Step 1: Inventory confirmed clips ----------------------------------
if not os.path.exists(CONFIRMED_ROOT):
    raise RuntimeError(f"No confirmed directory at {CONFIRMED_ROOT}. "
                       "Run Cell 5b (Workflow A) and/or label Workflow B clips first.")

confirmed_counts = {}
for true_label in sorted(os.listdir(CONFIRMED_ROOT)):
    n = count_clips_in(os.path.join(CONFIRMED_ROOT, true_label))
    if n:
        confirmed_counts[true_label] = n

if not confirmed_counts:
    raise RuntimeError("confirmed/ is empty. Run Cell 5b or label Workflow B clips first.")

print("Confirmed clips ready for training:")
for label, n in confirmed_counts.items():
    tag = "  ← FP corrections (weighted)" if label == "normal" else ""
    print(f"  confirmed/{label:<15}  {n} clips{tag}")


# ---- Step 2: Sequence extraction helpers --------------------------------

def get_clip_duration_s(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0.0
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    return frames / fps


def extract_all_frames_kp(video_path):
    cap    = cv2.VideoCapture(video_path)
    all_kp = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        all_kp.append(extract_keypoints(frame))
    cap.release()
    return all_kp


def extract_sequences_universal(video_path,
                                 n_frames=N_FRAMES,
                                 overlap=SLIDING_OVERLAP,
                                 threshold_s=SHORT_CLIP_THRESHOLD_S):
    duration_s = get_clip_duration_s(video_path)
    if duration_s <= 0:
        return [], "error"

    if duration_s <= threshold_s:
        cap   = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total < 2:
            cap.release()
            return [], "too_short"
        indices  = np.linspace(0, total - 1, n_frames, dtype=int)
        sequence = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if not ret:
                sequence.append(np.zeros(FEATURES, dtype=np.float32))
            else:
                sequence.append(extract_keypoints(frame))
        cap.release()
        if len(sequence) != n_frames:
            return [], "incomplete"
        return [np.stack(sequence)], "sampled"
    else:
        all_kp = extract_all_frames_kp(video_path)
        if len(all_kp) < n_frames:
            return [], "too_short"
        step   = n_frames - overlap
        chunks = []
        for start in range(0, len(all_kp) - n_frames + 1, step):
            chunks.append(np.stack(all_kp[start : start + n_frames]))
        return chunks, "sliding_window"


def load_dir_as_sequences(directory, label_idx, desc, repeat=1):
    seqs, labels = [], []
    clips = sorted([f for f in Path(directory).rglob("*")
                    if f.suffix.lower() in VID_EXTS])
    n_sampled, n_sliding, n_skipped, n_windows = 0, 0, 0, 0

    for clip in clips:
        windows, strategy = extract_sequences_universal(str(clip))
        if not windows:
            n_skipped += 1
            continue
        if strategy == "sampled":
            n_sampled += 1
        else:
            n_sliding += 1
        n_windows += len(windows)
        for window in windows:
            for _ in range(repeat):
                seqs.append(window)
                labels.append(label_idx)

    repeat_note = f" (x{repeat} each)" if repeat > 1 else ""
    print(
        f"  {desc:<40}  "
        f"{n_sampled:>3} short  "
        f"{n_sliding:>3} long  "
        f"{n_skipped:>2} skip  "
        f"-> {n_windows:>4} windows{repeat_note}"
    )
    return seqs, labels


# ── Augmentation (mirrors training notebook exactly) ─────────────────────────
def augment_sequence(seq: np.ndarray) -> np.ndarray:
    n, f = seq.shape
    aug  = seq.copy().reshape(n, N_KEYPOINTS, 3)
    if random.random() > 0.5:
        aug[:, :, 0] = -aug[:, :, 0]
    if random.random() > 0.5:
        skip = random.randint(1, 2)
        kept = sorted(random.sample(range(n), n - skip))
        aug  = np.concatenate([aug[kept],
                               np.tile(aug[kept[-1]], (skip, 1, 1))])
    aug += np.random.normal(0, 0.005, aug.shape).astype(np.float32)
    return aug.reshape(n, f)


# ---- Step 3: Resolve class list (same order as original training) --------
original_classes = sorted([
    d.name for d in Path(SOURCE_DATA_ROOT).iterdir()
    if d.is_dir() and not d.name.startswith(".")
])
print(f"\nOriginal class list: {original_classes}")

all_seqs, all_labels = [], []


# ---- Step 4: Load confirmed/ — one loop, handles both workflows ----------
# FP corrections (fp_ prefix) in confirmed/normal/ get FP_WEIGHT_FACTOR repeats.
# Everything else (src_ ground-truth and TP clip copies) gets repeat=1.
print("\nLoading confirmed/ fine-tuning pool...")
print(f"  (Short-clip threshold: {SHORT_CLIP_THRESHOLD_S}s)")

for true_label in sorted(os.listdir(CONFIRMED_ROOT)):
    if true_label not in original_classes:
        print(f"  [WARN] '{true_label}' not in original_classes {original_classes} — skipping.")
        continue

    cls_idx   = original_classes.index(true_label)
    label_dir = os.path.join(CONFIRMED_ROOT, true_label)

    # Split clips by prefix to apply per-clip weighting
    all_clips  = sorted([f for f in Path(label_dir).rglob("*")
                         if f.suffix.lower() in VID_EXTS])
    fp_clips   = [c for c in all_clips if c.name.startswith("fp_")]
    base_clips = [c for c in all_clips if not c.name.startswith("fp_")]

    # Regular clips (ground-truth src_ + confirmed Workflow B TPs)
    if base_clips:
        seqs, labels_list = [], []
        n_sampled, n_sliding, n_skipped, n_windows = 0, 0, 0, 0
        for clip in base_clips:
            windows, strategy = extract_sequences_universal(str(clip))
            if not windows:
                n_skipped += 1
                continue
            (n_sampled if strategy == "sampled" else n_sliding).__add__(1)
            n_windows += len(windows)
            for w in windows:
                seqs.append(w)
                labels_list.append(cls_idx)
        all_seqs.extend(seqs)
        all_labels.extend(labels_list)
        print(f"  confirmed/{true_label:<13} (base)  "
              f"{len(base_clips)} clips  -> {len(seqs)} windows")

    # FP correction clips in normal/ — weighted
    if fp_clips:
        seqs, labels_list = [], []
        for clip in fp_clips:
            windows, strategy = extract_sequences_universal(str(clip))
            if not windows:
                continue
            for w in windows:
                for _ in range(FP_WEIGHT_FACTOR):
                    seqs.append(w)
                    labels_list.append(cls_idx)
        all_seqs.extend(seqs)
        all_labels.extend(labels_list)
        print(f"  confirmed/{true_label:<13} (FP)    "
              f"{len(fp_clips)} clips  -> {len(seqs)} windows (x{FP_WEIGHT_FACTOR} each)")


X = np.stack(all_seqs)
y = np.array(all_labels)
n_classes = len(original_classes)

print(f"\nFull dataset: {len(X)} sequences across {n_classes} classes")
for i, name in enumerate(original_classes):
    print(f"  [{i}] {name:<15}  {(y == i).sum():>5} sequences")

## Cell 9 — Fine-tune and save

In [ ]:
# ── Stratified split ──────────────────────────────────────────────────────────
def stratified_split(X, y, val_frac, seed):
    rng = np.random.default_rng(seed)
    tr_idx, vl_idx = [], []
    for cls in np.unique(y):
        idx   = np.where(y == cls)[0]
        idx   = rng.permutation(idx)
        n_val = max(1, int(len(idx) * val_frac))
        vl_idx.extend(idx[:n_val])
        tr_idx.extend(idx[n_val:])
    return X[tr_idx], X[vl_idx], y[tr_idx], y[vl_idx]

X_train, X_val, y_train, y_val = stratified_split(X, y, FT_VAL_SPLIT, SEED)

print(f"Train: {len(X_train)}  Val: {len(X_val)}")
print(f"  Train distribution: {dict(Counter(y_train))}")
print(f"  Val   distribution: {dict(Counter(y_val))}")

# ── Augmentation ─────────────────────────────────────────────────────────────
aug_seqs, aug_labels = [], []
for seq, label in zip(X_train, y_train):
    aug_seqs.append(augment_sequence(seq))
    aug_labels.append(label)
    if original_classes[label] != "normal":
        for _ in range(MINORITY_COPIES):
            aug_seqs.append(augment_sequence(seq))
            aug_labels.append(label)

X_train = np.concatenate([X_train, np.stack(aug_seqs)])
y_train  = np.concatenate([y_train, np.array(aug_labels)])

perm    = np.random.permutation(len(X_train))
X_train = X_train[perm]
y_train  = y_train[perm]

print("After augmentation:")
print(f"  Train  : {len(X_train)}  Val: {len(X_val)} (val unchanged)")
print(f"  Distribution: {dict(Counter(y_train))}")

# ── Dynamic class weights ─────────────────────────────────────────────────────
counts       = Counter(y_train)
total        = len(y_train)
class_weights = torch.tensor(
    [total / (n_classes * counts[i]) for i in range(n_classes)],
    dtype=torch.float32,
).to(DEVICE)
print("\nClass weights:")
for i, name in enumerate(original_classes):
    print(f"  {name:<15}  {class_weights[i]:.3f}")

# ── Data loaders ──────────────────────────────────────────────────────────────
X_tr_t = torch.tensor(X_train, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.long)
X_vl_t = torch.tensor(X_val,   dtype=torch.float32)
y_vl_t = torch.tensor(y_val,   dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                          batch_size=FT_BATCH_SIZE, shuffle=True,
                          pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(TensorDataset(X_vl_t, y_vl_t),
                          batch_size=FT_BATCH_SIZE,
                          pin_memory=torch.cuda.is_available())

# ── Model — initialise from checkpoint (fine-tune, not scratch) ───────────────
model = PoseActivityClassifier(
    ckpt["features"], ckpt["hidden1"], ckpt["hidden2"],
    n_classes, dropout=0.3,
).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
print(f"\nStarting from checkpoint val_acc={ckpt.get('val_accuracy', 0):.1%}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=FT_LR_PATIENCE, verbose=False
)

best_val_acc       = 0.0
best_state         = None
early_stop_counter = 0
history            = {"train_loss": [], "val_acc": [], "lr": []}

print(f"Fine-tuning — up to {FT_EPOCHS} epochs  (patience={FT_EARLY_STOP})")
print(f"{'Epoch':>6}  {'Loss':>8}  {'Val acc':>8}  {'LR':>9}")
print("-" * 40)

for epoch in range(1, FT_EPOCHS + 1):

    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
    avg_loss = total_loss / len(X_train)

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct += (model(xb).argmax(dim=1) == yb).sum().item()
    val_acc = correct / len(X_val)

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_acc)
    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc       = val_acc
        best_state         = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    if epoch % 5 == 0 or improved:
        marker = "  ← best" if improved else ""
        print(f"{epoch:>6}  {avg_loss:>8.4f}  {val_acc:>7.1%}  "
              f"{current_lr:>9.2e}{marker}")

    if early_stop_counter >= FT_EARLY_STOP:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nFine-tuning complete.")
print(f"  Previous val_acc : {ckpt.get('val_accuracy', 0):.1%}")
print(f"  New val_acc      : {best_val_acc:.1%}")

# ── Swap the live CLASSIFIER to the fine-tuned weights ───────────────────────
# This makes the fine-tuned model available for Cell 9b regression tests
# without needing to reload from disk.
CLASSIFIER.load_state_dict(best_state)
CLASSIFIER.eval()
print("\nIn-session CLASSIFIER updated. Cell 9b will use the fine-tuned model.")

# ── Save fine-tuned checkpoint ────────────────────────────────────────────────
model.load_state_dict(best_state)
model.eval()

NEW_MODEL_PATH = "/kaggle/working/activity_classifier_v2.pt"
torch.save(
    {
        "model_state":    best_state,
        "class_names":    original_classes,
        "n_frames":       N_FRAMES,
        "features":       ckpt["features"],
        "hidden1":        ckpt["hidden1"],
        "hidden2":        ckpt["hidden2"],
        "architecture":   "bidirectional_gru",
        "val_accuracy":   best_val_acc,
        "fine_tuned":     True,
        "confirmed_clips_used": confirmed_counts,
        "history":            history,
    },
    NEW_MODEL_PATH,
)

print(f"\nSaved → {NEW_MODEL_PATH}")
print(">>> Download from the Output tab and upload to your cbms-classifier dataset <<<")

## Cell 9b — Workflow B: post-retrain TP regression test

After fine-tuning, this cell runs the updated model on every clip in `confirmed/<active_label>/` that came from Workflow B (i.e. clips **without** the `src_` prefix and without the `fp_` prefix — these are the TPs you confirmed during review).

**Why this matters (Q4):** A fine-tune that corrects false positives could accidentally regress on true positives if the FP corrections are very similar to real behaviours. This cell catches those regressions early.

### What it outputs

| Result | Meaning | Action |
|---|---|---|
| ✅ PASS | Model still confidently predicts the correct label | No action needed |
| ⚠ REGRESSION | Confidence < `TP_REGRESSION_CONFIDENCE` or wrong label | Clip is flagged |

Flagged clips are copied to `regression_flagged/<label>/` and listed in `regression_report.json`. You can optionally feed them back into another fine-tune round by running Cell 6b–9 again after moving them to `confirmed/<label>/`.

> Set `TP_REGRESSION_CONFIDENCE` in Cell 3 (default 0.70). Clips that score above this on the correct class are considered passing.

In [ ]:
REGRESSION_DIR    = "/kaggle/working/regression_flagged"
REGRESSION_REPORT = "/kaggle/working/regression_report.json"

regression_results = []
passed  = 0
flagged = 0
total   = 0

print("[Workflow B] Post-retrain TP regression test")
print(f"  Confidence threshold : {TP_REGRESSION_CONFIDENCE:.0%}")
print(f"  Using model          : fine-tuned in this session")
print()

for true_label in sorted(os.listdir(CONFIRMED_ROOT)):
    if true_label == "normal":
        # FP-corrected 'normal' clips are not TP clips — skip
        continue
    if true_label not in original_classes:
        continue

    label_dir = Path(CONFIRMED_ROOT) / true_label
    # Test only Workflow B TP clips (no src_ prefix, no fp_ prefix)
    tp_clips = [
        f for f in label_dir.rglob("*")
        if f.suffix.lower() in VID_EXTS
        and not f.name.startswith("src_")
        and not f.name.startswith("fp_")
    ]

    if not tp_clips:
        continue

    print(f"  Testing confirmed/{true_label}/  ({len(tp_clips)} Workflow B TP clips)")

    for clip in sorted(tp_clips):
        predicted, conf, probs = classify_clip_file(str(clip))
        correct_conf = probs.get(true_label, 0.0)
        total += 1

        is_regression = (
            predicted != true_label
            or correct_conf < TP_REGRESSION_CONFIDENCE
        )

        result_entry = {
            "clip":          str(clip),
            "true_label":    true_label,
            "predicted":     predicted,
            "correct_conf":  correct_conf,
            "all_probs":     probs,
            "regression":    is_regression,
        }
        regression_results.append(result_entry)

        if is_regression:
            flagged += 1
            # Copy to regression_flagged/<label>/
            flag_dest = Path(REGRESSION_DIR) / true_label
            flag_dest.mkdir(parents=True, exist_ok=True)
            shutil.copy2(clip, flag_dest / clip.name)
            print(f"    ⚠ REGRESSION  {clip.name}")
            print(f"       true={true_label}  predicted={predicted}  "
                  f"conf_on_true={correct_conf:.0%}")
        else:
            passed += 1

print()
print(f"Regression test complete.")
print(f"  Total TP clips tested : {total}")
print(f"  Passed (✅)           : {passed}")
print(f"  Flagged regressions   : {flagged}")

if flagged > 0:
    print(f"\n  Flagged clips copied to: {REGRESSION_DIR}")
    print(f"  Review them and optionally move back to confirmed/<label>/")
    print(f"  then re-run Cells 6b–9 for another fine-tune round.")
else:
    print(f"\n  No regressions detected. Fine-tuned model is stable on all TP clips.")

# Save report
with open(REGRESSION_REPORT, "w") as f:
    json.dump({
        "total_tested":   total,
        "passed":         passed,
        "flagged":        flagged,
        "threshold":      TP_REGRESSION_CONFIDENCE,
        "model_val_acc":  best_val_acc,
        "results":        regression_results,
    }, f, indent=2)
print(f"\nFull report saved → {REGRESSION_REPORT}")

## Cell 10 — Post-retrain confusion matrix

Run after Cell 9 to see per-class accuracy on the validation set.

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        probs = F.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy()
        all_preds.extend(probs.argmax(axis=1))
        all_true.extend(yb.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
col_w     = 10

print(f"Confusion matrix — fine-tuned model  (val_acc={best_val_acc:.1%})")
print(f"{'':>14}", end="")
for name in original_classes:
    print(f"  {name[:col_w]:>{col_w}}", end="")
print()
for i, true_name in enumerate(original_classes):
    print(f"  {true_name[:12]:<12}", end="")
    for j in range(n_classes):
        count = ((all_true == i) & (all_preds == j)).sum()
        print(f"  {count:>{col_w}}", end="")
    print()

print()
for i, name in enumerate(original_classes):
    mask = all_true == i
    if mask.sum() == 0:
        continue
    acc = (all_preds[mask] == i).mean()
    print(f"  {name:<15}  {acc:.1%}  ({mask.sum()} samples)")

print(f"\n  Overall: {(all_preds == all_true).mean():.1%}")

## Cell 11 — Fine-tune training curve

Run after Cell 9 to plot loss and val accuracy across fine-tuning epochs.

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = len(history["train_loss"])
xs = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(xs, history["train_loss"], color="steelblue", linewidth=1.5)
ax1.set_title("Fine-tune training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(xs, [v * 100 for v in history["val_acc"]], color="seagreen", linewidth=1.5)
ax2.axhline(best_val_acc * 100, color="tomato", linestyle="--",
            linewidth=1, label=f"Best: {best_val_acc:.1%}")
ax2.axhline(ckpt.get("val_accuracy", 0) * 100, color="gray", linestyle=":",
            linewidth=1, label=f"Prev: {ckpt.get('val_accuracy', 0):.1%}")
ax2.set_title("Fine-tune validation accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/finetune_curve.png", dpi=120)
plt.show()
print("Saved -> finetune_curve.png")

## Cell 12 — NTU RGB+D skeleton integration (optional)

Downloads the pre-processed NTU60 3D skeleton pkl from MMAction2 and extracts the **throwing** action class (A030), which is the closest proxy for a littering gesture in the dataset.

**Setup:**
1. Download `ntu60_3d.pkl` from `https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_3d.pkl`
2. Upload it to a Kaggle dataset and attach to this notebook.
3. Set `NTU_PKL_PATH` below and run this cell.

> Skip this cell entirely if `NTU_PKL_PATH = ""` — it prints a notice and exits cleanly.

In [ ]:
NTU_PKL_PATH           = "/kaggle/working/ntu60_3d.pkl.2"   # set "" to skip
NTU_THROWING_ACTION_ID = 7    # A007 = throwing in NTU60
NTU_MAX_CLIPS          = 200

"""
NTU RGB+D 3D skeleton -> CBMS training sequences.
(See previous notebook version for full joint mapping documentation.)
"""

if not NTU_PKL_PATH or not os.path.exists(NTU_PKL_PATH):
    print(f"NTU_PKL_PATH not set or not found: {NTU_PKL_PATH}")
    print("Skipping NTU integration. Set NTU_PKL_PATH above to enable.")
else:
    import pickle

    NTU_TO_MP = {
        3:  0,  4: 11,  5: 13,  6: 15,
        8: 12,  9: 14, 10: 16,
       12: 23, 13: 25, 14: 27,
       16: 24, 17: 26, 18: 28,
    }

    def ntu_skeleton_to_mp_vec(skeleton_seq, n_frames=N_FRAMES):
        T = skeleton_seq.shape[0]
        if T < 2:
            return None
        indices = np.linspace(0, T - 1, n_frames, dtype=int)
        result  = []
        for t in indices:
            frame_joints = skeleton_seq[t]
            mp_frame = np.zeros((33, 3), dtype=np.float32)
            for ntu_j, mp_j in NTU_TO_MP.items():
                mp_frame[mp_j] = frame_joints[ntu_j]
            nose_x, nose_y = mp_frame[0, 0], mp_frame[0, 1]
            mp_frame[:, 0] -= nose_x
            mp_frame[:, 1] -= nose_y
            result.append(mp_frame.flatten())
        return np.stack(result)

    print(f"Loading NTU pkl: {NTU_PKL_PATH} ...")
    with open(NTU_PKL_PATH, "rb") as f:
        ntu_data = pickle.load(f)

    annotations  = ntu_data.get("annotations", [])
    TARGET_LABEL = NTU_THROWING_ACTION_ID - 1

    ntu_seqs, ntu_labels = [], []
    skipped = 0

    TARGET_CLASS_IDX = original_classes.index("littering") if "littering" in original_classes else None

    if TARGET_CLASS_IDX is not None:
        for ann in annotations:
            if ann.get("label") != TARGET_LABEL or len(ntu_seqs) >= NTU_MAX_CLIPS:
                continue
            kp = ann.get("keypoint")
            if kp is None or kp.ndim == 4 and kp.shape[0] > 0:
                if kp.ndim == 4:
                    kp = kp[0]
            if kp is None or kp.shape[1] != 25:
                skipped += 1
                continue
            seq = ntu_skeleton_to_mp_vec(kp, N_FRAMES)
            if seq is None:
                skipped += 1
                continue
            ntu_seqs.append(seq)
            ntu_labels.append(TARGET_CLASS_IDX)

        print(f"  Throwing clips extracted : {len(ntu_seqs)}")
        print(f"  Skipped                  : {skipped}")

        if ntu_seqs:
            X = np.concatenate([X, np.stack(ntu_seqs)])
            y = np.concatenate([y, np.array(ntu_labels)])
            print(f"\nAfter NTU merge:")
            for i, name in enumerate(original_classes):
                print(f"  [{i}] {name:<15}  {(y == i).sum():>5} sequences")
            print("\nRun Cell 9 to fine-tune with NTU data included.")